# TD - CaSSaD

## Exercice 2

In [ ]:
import tensorflow as tf

In [ ]:
(x_full, y_full), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

In [ ]:
x_full, x_test = x_full.astype("float32") / 255.0, x_test.astype("float32") / 255.0
x_train, x_val = x_full[:-5000], x_full[-5000:]
y_train, y_val = y_full[:-5000], y_full[-5000:]

In [ ]:
def plain_mlp(depth=20, batch_norm=False):
  layers = [tf.keras.Input(shape=(32, 32, 3)), tf.keras.layers.Flatten()]
  for _ in range(depth):
    if batch_norm:
      layers += [tf.keras.layers.Dense(100, use_bias=False),
      tf.keras.layers.BatchNormalization(),
      tf.keras.layers.Activation("silu")]
    else:
      layers.append(tf.keras.layers.Dense(100, activation="silu"))
      layers.append(tf.keras.layers.Dense(10, activation="softmax"))
  return tf.keras.Sequential(layers)

In [ ]:
def fit(model, optimizer, epochs=200):
  model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
  early = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
  return model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=epochs,
  batch_size=128, callbacks=[early], verbose=0)

### Question 1

In [ ]:
# Whe shoud have 500 210 parameters

### Question 2

In [ ]:
import numpy as np
import tensorflow as tf

optimizers = {
    "SGD": lambda: tf.keras.optimizers.SGD(1e-2),
    "Momentum": lambda: tf.keras.optimizers.SGD(1e-2, momentum=0.9),
    "Nesterov": lambda: tf.keras.optimizers.SGD(1e-2, momentum=0.9, nesterov=True),
    "RMSProp": lambda: tf.keras.optimizers.RMSprop(1e-3),
    "Adam": lambda: tf.keras.optimizers.Adam(1e-3),
    "Nadam": lambda: tf.keras.optimizers.Nadam(1e-3),
}

histories = {}
for name, make_optimizer in optimizers.items():
    tf.keras.utils.set_random_seed(42)  # same initialization for all optimizers
    histories[name] = fit(plain_mlp(), make_optimizer())
    acc = histories[name].history["val_accuracy"]
    print(name, max(acc), int(np.argmax(acc)) + 1)


### Question 3

In [ ]:
import numpy as np
import tensorflow as tf

optimizers = {
    "SGD": lambda: tf.keras.optimizers.SGD(1e-2),
    "Momentum": lambda: tf.keras.optimizers.SGD(1e-3, momentum=0.9),
}

histories = {}
for name, make_optimizer in optimizers.items():
    tf.keras.utils.set_random_seed(42)  # same initialization for all optimizers
    histories[name] = fit(plain_mlp(), make_optimizer())
    acc = histories[name].history["val_accuracy"]
    print(name, max(acc), int(np.argmax(acc)) + 1)

### Question 4

In [ ]:
import numpy as np
import tensorflow as tf

optimizers = {
    "Momentum_lr1": lambda: tf.keras.optimizers.SGD(1e-4, momentum=0.9),
    "Momentum_lr2": lambda: tf.keras.optimizers.SGD(3 * 1e-4, momentum=0.9),
    "Momentum_lr3": lambda: tf.keras.optimizers.SGD(1e-3, momentum=0.9),
    "Momentum_lr4": lambda: tf.keras.optimizers.SGD(3 * 1e-3, momentum=0.9),
    "Momentum_lr5": lambda: tf.keras.optimizers.SGD(1e-2, momentum=0.9),
}

histories = {}
for name, make_optimizer in optimizers.items():
    tf.keras.utils.set_random_seed(42)  # same initialization for all optimizers
    histories[name] = fit(plain_mlp(), make_optimizer())
    acc = histories[name].history["val_accuracy"]
    print(name, max(acc), int(np.argmax(acc)) + 1)


### Question 5

In [ ]:
import numpy as np
import tensorflow as tf

optimizers = {
    "SGD": lambda: tf.keras.optimizers.SGD(1e-2),
    "Momentum": lambda: tf.keras.optimizers.SGD(1e-2, momentum=0.9),
    "Nesterov": lambda: tf.keras.optimizers.SGD(1e-2, momentum=0.9, nesterov=True),
    "RMSProp": lambda: tf.keras.optimizers.RMSprop(1e-3),
    "Adam": lambda: tf.keras.optimizers.Adam(1e-3),
    "Nadam": lambda: tf.keras.optimizers.Nadam(1e-3),
}

histories = {}
for name, make_optimizer in optimizers.items():
    tf.keras.utils.set_random_seed(42)  # same initialization for all optimizers
    histories[name] = fit(plain_mlp(), make_optimizer())
    acc = histories[name].history["val_accuracy"]
    print(name, max(acc), int(np.argmax(acc)) + 1)


### Question 6

In [ ]:
import tensorflow as tf


def residual_mlp(n_blocks=9, activation_after_add=True):
    inputs = tf.keras.Input(shape=(32, 32, 3))
    x = tf.keras.layers.Flatten()(inputs)
    x = tf.keras.layers.Dense(100, activation="silu")(x)
    for _ in range(n_blocks):
        f = tf.keras.layers.Dense(100, activation="silu")(x)
        f = tf.keras.layers.Dense(100)(f)
        x = tf.keras.layers.Add()([f, x])
        if activation_after_add:
            x = tf.keras.layers.Activation("silu")(x)
    outputs = tf.keras.layers.Dense(10, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

### Question 7

In [ ]:
import numpy as np
import tensorflow as tf

(x_full, y_full), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_full, x_test = x_full.astype("float32") / 255.0, x_test.astype("float32") / 255.0
x_train, x_val = x_full[:-5000], x_full[-5000:]
y_train, y_val = y_full[:-5000], y_full[-5000:]

x_batch, y_batch = x_train[:128], y_train[:128]
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()


def kernel_grad_norms(model):
    kernels = [w for w in model.trainable_variables if w.name == "kernel"][
        :-1
    ]  # hidden layers
    with tf.GradientTape() as tape:
        loss = loss_fn(y_batch, model(x_batch, training=True))
    return [float(tf.norm(g)) for g in tape.gradient(loss, kernels)]


for name, model in [
    ("plain", plain_mlp()),
    ("batch norm", plain_mlp(batch_norm=True)),
    ("residual", residual_mlp()),
]:
    print(name, np.round(kernel_grad_norms(model), 8))


## Exercic 4

### Question 1

In [ ]:
I = tf.constant([[2, 1, 3, 0], [1, 1, 0, 5], [3, 3, 1, 0], [2, 0, 0, 2]], tf.float32)
K = tf.constant([[1, 0, 2], [2, 1, 0], [1, 0, 3]], tf.float32)
x = tf.reshape(I, (1, 4, 4, 1)) # (batch, height, width, channels)
k = tf.reshape(K, (3, 3, 1, 1)) # (height, width, in_channels, out_channels)

for padding in ("VALID", "SAME"):
  xcorr = tf.nn.conv2d(x, k, strides=1, padding=padding)
  conv = tf.nn.conv2d(x, tf.reverse(k, axis=[0, 1]), strides=1, padding=padding)
  print(padding, tf.squeeze(xcorr).numpy(), tf.squeeze(conv).numpy(), sep="\n")

### Question 2

In [ ]:
(x_tr, _), _ = tf.keras.datasets.fashion_mnist.load_data()
X = tf.constant(x_tr[0], tf.float32)[None, :, :, None] / 255.0
X = tf.pad(X, [[0, 0], [10, 10], [10, 10], [0, 0]])
phi = tf.keras.layers.Conv2D(8, 3, strides=1, padding="same", bias_initializer="random_normal")
T = lambda Z, tau: tf.roll(Z, shift=tau, axis=[1, 2])

for tau in [(1, 0), (2, 3), (5, -4)]:
  print(tau, float(tf.reduce_max(tf.abs(phi(T(X, tau)) - T(phi(X), tau)))))

### Question 3

In [ ]:
phi_s = tf.keras.layers.Conv2D(8, 3, strides=2, padding="same")
for tau in [(2, 0), (4, -6), (1, 0), (3, 2)]:
  delta = (tau[0] // 2, tau[1] // 2)
  print(tau, float(tf.reduce_max(tf.abs(phi_s(T(X, tau)) - T(phi_s(X), delta)))))

conv_pool = tf.keras.Sequential([tf.keras.layers.Conv2D(8, 3, padding="same"), tf.keras.layers.MaxPooling2D(2)])

### Question 4

In [ ]:
(x_full, y_full), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
prep = lambda x: tf.pad(x[..., None].astype("float32") / 255.0, [[0, 0], [2, 2], [2, 2], [0, 0]])
x_full, x_test = prep(x_full), prep(x_test) # 32 x 32 x 1
x_train, x_val, y_train, y_val = x_full[:-10000], x_full[-10000:], y_full[:-10000], y_full[-10000:]


def lenet5(activation="tanh", pooling="avg", batch_norm=False, dropout=0.0, augment=False):
  Pool = tf.keras.layers.AveragePooling2D if pooling == "avg" else tf.keras.layers.MaxPooling2D
  layers = [tf.keras.Input(shape=(32, 32, 1))]
  if augment:
    layers.append(tf.keras.layers.RandomTranslation(2 / 32, 2 / 32, fill_mode="constant"))
  for filters, pool in ((6, True), (16, True), (120, False)):
    layers.append(tf.keras.layers.Conv2D(filters, 5))
    if batch_norm:
      layers.append(tf.keras.layers.BatchNormalization())
    layers.append(tf.keras.layers.Activation(activation))
    if pool:
      layers.append(Pool(pool_size=2, strides=2))
  layers += [tf.keras.layers.Flatten(), tf.keras.layers.Dense(84, activation=activation)] # F6
  if dropout:
    layers.append(tf.keras.layers.Dropout(dropout))
  layers.append(tf.keras.layers.Dense(10, activation="softmax"))
  return tf.keras.Sequential(layers)

### Question 6

In [ ]:
configs = {
  "LeNet-5": dict(),
  "+ ReLU": dict(activation="relu"),
  "+ max pooling": dict(activation="relu", pooling="max"),
  "+ batch norm": dict(activation="relu", pooling="max", batch_norm=True),
  "+ dropout": dict(activation="relu", pooling="max", batch_norm=True, dropout=0.5),
  "+ augmentation": dict(activation="relu", pooling="max", batch_norm=True, dropout=0.5, augment=True),
}

for name, kwargs in configs.items():
  tf.keras.utils.set_random_seed(42)
  model = lenet5(**kwargs)
  model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
  early = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)
  h = model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=50, batch_size=128,
  callbacks=[early], verbose=0)
  print(name, model.count_params(), max(h.history["val_accuracy"]))

## Question 7

In [ ]:
def shift_right(x, k): # zeros enter on the left
  return tf.pad(x, [[0, 0], [0, 0], [k, 0], [0, 0]])[:, :, :32, :]

for name, model in trained_models.items(): # dictionary of the models trained above
  print(name, [model.evaluate(shift_right(x_val, k), y_val, verbose=0)[1] for k in range(5)])